# Figure 3 | AHBA sampling and transcriptomic enrichment

The surface panel displays the AHBA samples retained inside the participant 07 CORnet-S-MPNet encoding mask. Enrichment panels summarize competitive tests for GO Biological Process, adult human cortical cell-type markers and MitoCarta MitoPathways.

In [ ]:
import os
from pathlib import Path

import cortex
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from PIL import Image
from scipy.spatial import cKDTree

# Locate the analysis workspace. Set MITO_FMRI_ROOT explicitly when
# derivatives are stored outside this repository.
def find_analysis_root():
    configured = os.environ.get("MITO_FMRI_ROOT")
    candidates = [Path(configured)] if configured else []
    candidates += [Path.cwd(), Path.cwd() / "nsd_full_cortex", Path.cwd().parent / "nsd_full_cortex"]
    for candidate in candidates:
        if (candidate / "derivatives").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the analysis root. Set MITO_FMRI_ROOT to the nsd_full_cortex directory."
    )

ROOT = find_analysis_root()

AHBA_RESULTS = ROOT / 'derivatives' / 'ahba_enrichment'
SITE_TABLE = (
    ROOT / 'derivatives' / 'ahba_genomewide_gene_analysis' / 'cache'
    / 'left_cortical_sites_to_subject_masks.csv'
)
VECTORS = (
    ROOT / 'derivatives' / 'ahba_genomewide_gene_analysis' / 'cache'
    / 'gene_analysis_vectors_fsaverage10k.npz'
)
FSAVERAGE = ROOT / 'neuromaps_data' / 'atlases' / 'fsaverage'
SPHERE_10K = FSAVERAGE / 'tpl-fsaverage_den-10k_hemi-L_sphere.surf.gii'
SPHERE_164K = FSAVERAGE / 'tpl-fsaverage_den-164k_hemi-L_sphere.surf.gii'
OUTPUT = ROOT / 'derivatives' / 'figure3' / 'figures'
OUTPUT.mkdir(parents=True, exist_ok=True)

PYCORTEX_SUBJECT = 'fsaverage'
DISPLAY_SUBJECT = 7
DISPLAY_ANALYSIS = 'cornet_s_mpnet'
DISPLAY_MODEL_LABEL = 'CORnet-S–MPNet'
SITE_COLOR = "#EBEB09"

assert SITE_TABLE.exists()
assert VECTORS.exists()
assert SPHERE_10K.exists() and SPHERE_164K.exists()
assert PYCORTEX_SUBJECT in cortex.db.subjects

## Select the AHBA sites used for participant 07

In [ ]:
site_rows = pd.read_csv(
    SITE_TABLE,
    dtype={'donor': str, 'well_id': str},
)
used_sites = (
    site_rows[
        site_rows.subject.eq(DISPLAY_SUBJECT)
        & site_rows.analysis.eq(DISPLAY_ANALYSIS)
    ]
    .sort_values(['donor', 'well_id'])
    .reset_index(drop=True)
)
assert len(used_sites) == 169
assert used_sites.well_id.nunique() == 169
assert used_sites.donor.nunique() == 6

site_summary = pd.DataFrame({
    'subject': ['subj07'],
    'analysis': [DISPLAY_ANALYSIS],
    'n_AHBA_samples': [len(used_sites)],
    'n_unique_surface_vertices': [used_sites.nearest_vertex.nunique()],
    'n_donors': [used_sites.donor.nunique()],
    'maximum_surface_distance_mm': [used_sites.surface_distance_mm.max()],
})
display(site_summary)

## Project retained AHBA sites to the cortical surface

In [ ]:
sphere_10k = np.asarray(
    nib.load(SPHERE_10K).darrays[0].data, dtype=float
)
sphere_164k = np.asarray(
    nib.load(SPHERE_164K).darrays[0].data, dtype=float
)
sphere_10k /= np.linalg.norm(sphere_10k, axis=1, keepdims=True)
sphere_164k /= np.linalg.norm(sphere_164k, axis=1, keepdims=True)

_, vertex_10k_to_164k = cKDTree(sphere_164k).query(sphere_10k, k=1)
_, vertex_164k_to_10k = cKDTree(sphere_10k).query(sphere_164k, k=1)
used_sites['vertex_164k'] = vertex_10k_to_164k[
    used_sites.nearest_vertex.to_numpy(int)
]

flat_merged, _ = cortex.db.get_surf(
    PYCORTEX_SUBJECT, 'flat', merge=True, nudge=True
)
flat_merged = np.asarray(flat_merged)
n_vertices_hemi = sphere_164k.shape[0]
assert flat_merged.shape[0] == 2 * n_vertices_hemi
used_sites['flat_x'] = flat_merged[used_sites.vertex_164k, 0]
used_sites['flat_y'] = flat_merged[used_sites.vertex_164k, 1]

vectors = np.load(VECTORS)
mask_key = 'mask_subj07_cornet_s_mpnet_lh'
assert mask_key in vectors.files
mask_10k = vectors[mask_key].astype(bool)
mask_164k = mask_10k[vertex_164k_to_10k]

# Exact audit against the same 10k mask used to select downstream sites.
assert mask_10k[used_sites.nearest_vertex.to_numpy(int)].all()

print(f'AHBA samples displayed: {len(used_sites):,}')
print(f'Unique linked fsaverage10k vertices: {used_sites.nearest_vertex.nunique():,}')
print(f'Unique display vertices on fsaverage164k: {used_sites.vertex_164k.nunique():,}')
print(f'Left-hemisphere CORnet-S–MPNet mask vertices (10k): {mask_10k.sum():,}')

## Surface distribution of the 169 retained sites

In [ ]:
def make_subj07_cornet_mask_overlay():
    red = np.zeros(2 * n_vertices_hemi, dtype=np.uint8)
    green = np.zeros_like(red)
    blue = np.zeros_like(red)
    alpha = np.zeros_like(red)
    red[:n_vertices_hemi][mask_164k] = 128
    green[:n_vertices_hemi][mask_164k] = 128
    blue[:n_vertices_hemi][mask_164k] = 128
    alpha[:n_vertices_hemi][mask_164k] = 200
    return cortex.VertexRGB(
        red, green, blue, PYCORTEX_SUBJECT, alpha=alpha
    )


def render_subj07_cornet_analysis_sites():
    overlay = make_subj07_cornet_mask_overlay()
    fig = cortex.quickflat.make_figure(
        overlay,
        with_curvature=True,
        with_rois=False,
        with_labels=False,
        with_colorbar=False,
        with_borders=False,
        recache=False,
        nanmean=True,
        height=1500,
        curvature_brightness=0.88,
        curvature_contrast=0.16,
    )
    ax = fig.axes[0]
    left_flat = flat_merged[:n_vertices_hemi, :2]
    pad_x = 0.025 * np.ptp(left_flat[:, 0])
    pad_y = 0.025 * np.ptp(left_flat[:, 1])
    ax.set_xlim(left_flat[:, 0].min() - pad_x, left_flat[:, 0].max() + pad_x)
    ax.set_ylim(left_flat[:, 1].min() - pad_y, left_flat[:, 1].max() + pad_y)

    ax.scatter(
        used_sites.flat_x, used_sites.flat_y,
        s=200, c=SITE_COLOR,
        edgecolors='white', linewidths=3,
        alpha=0.92, zorder=20,
    )
    handle = Line2D(
        [0], [0], marker='o', linestyle='none', markersize=7,
        markerfacecolor=SITE_COLOR, markeredgecolor='white',
        markeredgewidth=0.7,
        label=f'{DISPLAY_MODEL_LABEL} (n={len(used_sites)})',
    )
    """fig.legend(
        handles=[handle], loc='lower center', ncol=1, frameon=False,
        bbox_to_anchor=(0.5, 0.012), fontsize=9,
    )
    fig.suptitle(
        f'AHBA samples used for subj07 {DISPLAY_MODEL_LABEL} analysis '
        f'(n={len(used_sites)})',
        fontsize=14, y=0.965,
    )"""
    stem = 'subj07_cornet_s_mpnet_ahba_sites_within_encoding_mask_lh'
    png = OUTPUT / f'{stem}.png'
    pdf = OUTPUT / f'{stem}.pdf'
    fig.savefig(
        png, dpi=450, bbox_inches='tight', pad_inches=0.06,
        facecolor='white',
    )
    fig.savefig(
        pdf, bbox_inches='tight', pad_inches=0.06,
        facecolor='white',
    )
    plt.close(fig)
    return png, pdf


subj07_sites_png, subj07_sites_pdf = render_subj07_cornet_analysis_sites()
print(subj07_sites_png)
display(Image.open(subj07_sites_png))

## Model-robust enrichment summaries

In [ ]:
import textwrap

ENRICHMENT = ROOT / 'derivatives' / 'ahba_enrichment'
ENRICHMENT_RESULTS = ENRICHMENT / 'gene_set_mannwhitney_bh_results.csv.gz'
GO_REPRESENTATIVES = ENRICHMENT / 'go_bp_representative_results.csv.gz'

enrichment = pd.read_csv(ENRICHMENT_RESULTS)
go_representatives = pd.read_csv(GO_REPRESENTATIVES)

QCOL = 'mannwhitney_q_bh_within_library_panel'
PCOL = 'mannwhitney_p_two_sided'
MODEL_ORDER = ('cornet_s_mpnet', 'dinov2_minilm')
MODEL_LABEL = {
    'cornet_s_mpnet': 'CORnet-S +MPNet',
    'dinov2_minilm': 'DINOv2 + MiniLM',
}
MAP_LABEL = {
    'unique_visual': 'Unique visual variance',
    'unique_semantic': 'Unique semantic variance',
}
MAP_COLOR = {
    'unique_visual': '#2369C8',
    'unique_semantic': '#DC3246',
}
MAIN_LIBRARIES = (
    ('GO Biological Process', go_representatives),
    ('MitoCarta 3.0 MitoPathways',
     enrichment[enrichment.library.eq('MitoCarta 3.0 MitoPathways')]),
)
CELL_TYPE_DATA = enrichment[
    enrichment.library.eq('Adult human cortical cell types')
].copy()

plt.rcParams['hatch.linewidth'] = 1.25


def select_replicated_terms(data, variance_map, n_terms):
    panel = data[data.variance_map.eq(variance_map)].copy()
    q_wide = panel.pivot(index='gene_set', columns='analysis', values=QCOL)
    effect_wide = panel.pivot(
        index='gene_set', columns='analysis', values='rank_biserial_effect'
    )
    complete = q_wide.notna().all(axis=1) & effect_wide.notna().all(axis=1)
    replicated = (
        complete
        & q_wide['cornet_s_mpnet'].lt(0.05)
        & q_wide['dinov2_minilm'].lt(0.05)
        & np.sign(effect_wide['cornet_s_mpnet']).eq(
            np.sign(effect_wide['dinov2_minilm'])
        )
    )
    robust_effect = effect_wide.mean(axis=1).abs()
    terms = robust_effect[replicated].sort_values(ascending=False).head(n_terms).index.tolist()
    return panel[panel.gene_set.isin(terms)].copy(), terms


def enrichment_bubble_size(q):
    # Stronger separation than the standalone enrichment notebook.
    strength = np.clip(-np.log10(np.maximum(np.asarray(q, dtype=float), 1e-12)), 0, 5)
    return 35 + 65 * strength ** 1.5


def draw_enrichment_panel(ax, panel, terms, color, xmax, title):
    y_lookup = {term: idx for idx, term in enumerate(terms)}
    if panel.empty:
        ax.text(
            0.5, 0.5, 'No gene sets met\nreplication criteria',
            transform=ax.transAxes, ha='center', va='center',
            color='#666666', fontsize=10,
        )
    for model, offset in zip(MODEL_ORDER, (0, 0)):
        rows = panel[panel.analysis.eq(model)].copy()
        y = rows.gene_set.map(y_lookup).to_numpy(float) + offset
        common = dict(
            x=rows.rank_biserial_effect, y=y,
            s=enrichment_bubble_size(rows[QCOL]), facecolors='none',
            edgecolors=color, linewidths=1.65, alpha=0.96, zorder=3,
        )
        if model == 'cornet_s_mpnet':
            ax.scatter(**common)
        else:
            ax.scatter(**common, hatch='////')
    ax.axvline(0, color='#8A8F98', linewidth=1, zorder=1)
    ax.set_yticks(
        range(len(terms)),
        ['\n'.join(textwrap.wrap(
            str(term).split(' | ', 1)[-1].replace('_', ' '), 34
        )) for term in terms],
    )
    if terms:
        ax.invert_yaxis()
    ax.set_xlim(-xmax, xmax)
    ax.grid(axis='x', color='#E2E5E9', linewidth=0.65, zorder=0)
    ax.tick_params(axis='y', length=0, labelsize=9.6)
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Rank-biserial enrichment effect', fontsize=12)


def add_enrichment_legend(fig, axis, ncol=6, bbox=(0.5, 1.5)):
    hollow = axis.scatter([], [], s=95, facecolors='none', edgecolors='#333333',
                          linewidths=1.6, label=MODEL_LABEL['cornet_s_mpnet'])
    hatched = axis.scatter([], [], s=95, facecolors='none', edgecolors='#333333',
                           linewidths=1.6, hatch='////',
                           label=MODEL_LABEL['dinov2_minilm'])
    size_handles = [
        axis.scatter([], [], s=enrichment_bubble_size(q), facecolors='none',
                     edgecolors='#666666', linewidths=1.2, label=f'q < {q:g}')
        for q in (0.05, 0.01, 0.001, 0.0001)
    ]
    fig.legend(
        handles=[hollow, hatched, *size_handles], frameon=False, ncol=ncol,
        loc='upper center', bbox_to_anchor=bbox, columnspacing=1.5,
        handletextpad=0.55,
    )


def plot_main_enrichment(variance_map):
    color = MAP_COLOR[variance_map]
    selected = []
    for library, data in MAIN_LIBRARIES:
        panel, terms = select_replicated_terms(data, variance_map, n_terms=6)
        selected.append((library, panel, terms))
    xmax = max(
        0.10,
        max(float(np.abs(panel.rank_biserial_effect).max())
            for _, panel, _ in selected if not panel.empty) * 1.16,
    )
    # Use identical, explicitly positioned axes for visual and semantic figures.
    # This prevents label length from changing the plotting-area geometry.
    fig = plt.figure(figsize=(12, 5))
    axes = [
        fig.add_axes([0.20, 0.13, 0.30, 0.68]),
        fig.add_axes([0.68, 0.13, 0.30, 0.68]),
    ]
    for ax, (library, panel, terms) in zip(axes, selected):
        draw_enrichment_panel(ax, panel, terms, color, xmax, library)
    add_enrichment_legend(fig, axes[0], bbox=(0.5, 0.95))
    fig.suptitle(f'{MAP_LABEL[variance_map]} enrichment',
                 fontsize=14, fontweight='bold', y=0.985)
    stem = f'{variance_map}_enrichment_horizontal_top6'
    png, pdf = OUTPUT / f'{stem}.png', OUTPUT / f'{stem}.pdf'
    fig.savefig(png, dpi=450, bbox_inches=None, pad_inches=0,
                transparent=True)
    fig.savefig(pdf, bbox_inches=None, pad_inches=0, transparent=True)
    plt.show()
    return png, pdf


def plot_cell_type_enrichment(variance_map):
    color = MAP_COLOR[variance_map]
    panel, terms = select_replicated_terms(
        CELL_TYPE_DATA, variance_map, n_terms=8
    )
    xmax = (max(0.10, float(np.abs(panel.rank_biserial_effect).max()) * 1.18)
            if not panel.empty else 0.10)
    fig, ax = plt.subplots(figsize=(7.6, 5.5), constrained_layout=True)
    draw_enrichment_panel(
        ax, panel, terms, color, xmax, ''
    )
    add_enrichment_legend(fig, ax, ncol=3, bbox=(0.5, 1.105))
    fig.suptitle(f'{MAP_LABEL[variance_map]} cell-type enrichment',
                 fontsize=14, fontweight='bold', y=1.245)
    stem = f'{variance_map}_cell_type_enrichment'
    png, pdf = OUTPUT / f'{stem}.png', OUTPUT / f'{stem}.pdf'
    fig.savefig(png, dpi=450, bbox_inches='tight', pad_inches=0.06,
                transparent=True)
    fig.savefig(pdf, bbox_inches='tight', pad_inches=0.06, transparent=True)
    plt.show()
    return png, pdf


visual_enrichment_png, visual_enrichment_pdf = plot_main_enrichment('unique_visual')
semantic_enrichment_png, semantic_enrichment_pdf = plot_main_enrichment('unique_semantic')
visual_cell_png, visual_cell_pdf = plot_cell_type_enrichment('unique_visual')
semantic_cell_png, semantic_cell_pdf = plot_cell_type_enrichment('unique_semantic')
print(visual_enrichment_png)
print(semantic_enrichment_png)
print(visual_cell_png)
print(semantic_cell_png)

## Optional combined visual-semantic comparison

In [ ]:
from matplotlib.lines import Line2D

COMBINED_MAP_ORDER = ('unique_visual', 'unique_semantic')
COMBINED_MAP_OFFSET = {'unique_visual': -0.16, 'unique_semantic': 0.16}
COMBINED_MODEL_OFFSET = {'dinov2_minilm': -0.052, 'cornet_s_mpnet': 0.052}


def combined_selected_terms(data, n_terms=6):
    selected_by_map = {}
    for variance_map in COMBINED_MAP_ORDER:
        _, selected_by_map[variance_map] = select_replicated_terms(
            data, variance_map, n_terms=n_terms
        )
    union = list(dict.fromkeys(
        selected_by_map['unique_visual'] + selected_by_map['unique_semantic']
    ))

    panel = data[
        data.gene_set.isin(union)
        & data.variance_map.isin(COMBINED_MAP_ORDER)
        & data.analysis.isin(MODEL_ORDER)
    ].copy()
    robust = (
        panel.groupby(['gene_set', 'variance_map'], observed=True)
        .rank_biserial_effect.mean().unstack('variance_map')
        .reindex(union)
    )
    robust['visual_semantic_separation'] = (
        robust.get('unique_semantic', np.nan)
        - robust.get('unique_visual', np.nan)
    )
    ordered_terms = robust.visual_semantic_separation.sort_values(
        ascending=False, na_position='last'
    ).index.tolist()
    return panel, ordered_terms


def draw_combined_enrichment_panel(ax, panel, terms, xmax, title):
    y_lookup = {term: idx for idx, term in enumerate(terms)}
    for variance_map in COMBINED_MAP_ORDER:
        for model in MODEL_ORDER:
            rows = panel[
                panel.variance_map.eq(variance_map)
                & panel.analysis.eq(model)
            ].copy()
            if rows.empty:
                continue
            rows['y'] = (
                rows.gene_set.map(y_lookup).astype(float)
                + COMBINED_MAP_OFFSET[variance_map]
                + COMBINED_MODEL_OFFSET[model]
            )
            for is_significant, alpha in ((True, 0.98), (False, 0.28)):
                subset = rows[rows[QCOL].lt(0.05).eq(is_significant)]
                if subset.empty:
                    continue
                common = dict(
                    x=subset.rank_biserial_effect,
                    y=subset.y,
                    s=enrichment_bubble_size(subset[QCOL]),
                    facecolors='none',
                    edgecolors=MAP_COLOR[variance_map],
                    linewidths=1.55,
                    alpha=alpha,
                    zorder=3,
                )
                if model == 'dinov2_minilm':
                    ax.scatter(**common)
                else:
                    ax.scatter(**common, hatch='////')

    ax.axvline(0, color='#7A7F87', linewidth=1.05, zorder=1)
    ax.set_yticks(
        range(len(terms)),
        ['\n'.join(textwrap.wrap(
            str(term).split(' | ', 1)[-1].replace('_', ' '), 30
        )) for term in terms],
    )
    ax.invert_yaxis()
    ax.set_xlim(-xmax, xmax)
    ax.grid(axis='x', color='#E2E5E9', linewidth=0.65, zorder=0)
    ax.tick_params(axis='y', length=0, labelsize=9.5)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Rank-biserial enrichment effect', fontsize=11)


def plot_combined_visual_semantic_enrichment():
    selected = []
    all_effects = []
    for library, data in MAIN_LIBRARIES:
        panel, terms = combined_selected_terms(data, n_terms=6)
        selected.append((library, panel, terms))
        all_effects.extend(panel.rank_biserial_effect.dropna().tolist())
    xmax = max(0.10, float(np.max(np.abs(all_effects))) * 1.13)

    fig = plt.figure(figsize=(12.0, 6.8))
    axes = [
        fig.add_axes([0.215, 0.13, 0.285, 0.675]),
        fig.add_axes([0.70, 0.13, 0.28, 0.675]),
    ]
    for ax, (library, panel, terms) in zip(axes, selected):
        draw_combined_enrichment_panel(ax, panel, terms, xmax, library)

    map_handles = [
        Line2D([], [], marker='o', linestyle='none', markersize=7,
               markerfacecolor='none', markeredgewidth=1.6,
               markeredgecolor=MAP_COLOR[m], label=MAP_LABEL[m])
        for m in COMBINED_MAP_ORDER
    ]
    model_handles = [
        axes[0].scatter([], [], s=85, facecolors='none', edgecolors='#333333',
                        linewidths=1.45, label=MODEL_LABEL['dinov2_minilm']),
        axes[0].scatter([], [], s=85, facecolors='none', edgecolors='#333333',
                        linewidths=1.45, hatch='////',
                        label=MODEL_LABEL['cornet_s_mpnet']),
    ]
    q_handles = [
        axes[0].scatter([], [], s=enrichment_bubble_size(q), facecolors='none',
                        edgecolors='#666666', linewidths=1.1, label=f'q = {q:g}')
        for q in (0.05, 0.01, 0.001, 0.0001)
    ]
    fig.legend(
        handles=[*map_handles, *model_handles],
        loc='upper center', bbox_to_anchor=(0.59, 0.925), ncol=4,
        frameon=False, columnspacing=1.15, handletextpad=0.45,
        fontsize=9,
    )
    fig.legend(
        handles=q_handles,
        loc='upper center', bbox_to_anchor=(0.59, 0.875), ncol=4,
        frameon=False, columnspacing=1.25, handletextpad=0.45,
        fontsize=8.8,
    )
    fig.suptitle(
        'Visual and semantic enrichment profiles',
        fontsize=14, fontweight='bold', y=0.982,
    )
    fig.text(
        0.59, 0.025,
        'Union of the visual and semantic top-six replicated terms; '
        'counterpart results with q >= 0.05 are shown with reduced opacity.',
        ha='center', va='center', fontsize=8.5, color='#555555',
    )
    stem = 'visual_semantic_combined_enrichment_top6_union'
    png = OUTPUT / f'{stem}.png'
    pdf = OUTPUT / f'{stem}.pdf'
    fig.savefig(png, dpi=450, bbox_inches=None, pad_inches=0,
                transparent=True)
    fig.savefig(pdf, bbox_inches=None, pad_inches=0, transparent=True)
    plt.show()
    return png, pdf


combined_enrichment_png, combined_enrichment_pdf = (
    plot_combined_visual_semantic_enrichment()
)
print(combined_enrichment_png)